In [2]:
!pip install cirq mitiq

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached protobuf-5.29.5-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 1.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 1.2 MB/s  0:00:

In [3]:
import cirq
import numpy as np
from mitiq import MeasurementResult, Observable, PauliString

In [4]:
def idle_qubits(circuit, qubits, idle_steps):
    """Set qubits to idle for specfied number of steps 
    in by inserting Identity gates in each `moment`
    """
    for step in range(idle_steps):
        circuit.append(cirq.I(q) for q in qubits)

    return circuit

In [5]:
def ghz(num_qubits, idle_steps=0):
    # Create  qubit registers
    qubits = cirq.LineQubit.range(num_qubits)

    # Create a quantum circuit
    circuit = cirq.Circuit()
    
    # Add CNOT gates to entangle the first qubit with each of the other qubits
    for i in range(num_qubits):
        if i == 0: 
            # Add a Hadamard gate to the first qubit
            circuit.append(cirq.H(qubits[0]))
        else:
            circuit.append(cirq.CNOT(qubits[0], qubits[i]))
            # Set qubits to idle for specfied number of steps
            # in the form of Identity gates
            other_qubits = qubits[1:i] + qubits[i+1:]
            circuit = idle_qubits(circuit, other_qubits, idle_steps)
    
    return circuit

In [6]:
num_qubits = 6
circuit = ghz(num_qubits, idle_steps=3)

print(circuit)

                  ┌──┐           ┌──┐           ┌──┐           ┌──┐
0: ───H───@────────@──────────────@──────────────@──────────────@─────────
          │        │              │              │              │
1: ───────X───I────┼I────I───I────┼I────I───I────┼I────I───I────┼I────I───
                   │              │              │              │
2: ───I───I───I────X─────I───I────┼I────I───I────┼I────I───I────┼I────────
                                  │              │              │
3: ───I───I───I────I─────I───I────X─────I───I────┼I────I───I────┼I────────
                                                 │              │
4: ───I───I───I────I─────I───I────I─────I───I────X─────I───I────┼I────────
                                                                │
5: ───I───I───I────I─────I───I────I─────I───I────I─────I───I────X─────────
                  └──┘           └──┘           └──┘           └──┘


In [7]:
def execute(
    circuit: cirq.Circuit, 
    rz_noise: float = 0.02,
    depolar_noise: float = 0.005
    ) -> MeasurementResult:
    """
    Execute a circuit with R_z dephasing noise of strength ``rz_noise`` and depolarizing noise ``depolar_noise``
    """
    # Simulate systematic dephasing (coherent RZ) on each qubit for each moment.
    circuit = circuit.with_noise(cirq.rz(rz_noise))

    # Simulate systematic depolarizing on each qubit for each moment.
    circuit = circuit.with_noise(cirq.bit_flip(depolar_noise))

    # Measure out all qubits
    circuit += cirq.measure(*sorted(circuit.all_qubits()), key="m")

    # Use a noise simulator
    simulator = cirq.DensityMatrixSimulator()

    # Run the circuit 1000 times
    result = simulator.run(circuit, repetitions=1000)

    # Retrieve the measurement results in bitstring form
    bitstrings = result.measurements["m"]

    return MeasurementResult(bitstrings)

In [8]:
res = execute(circuit)
res.to_dict() # Dictionary for more convenient visualization

{'nqubits': 6,
 'qubit_indices': (0, 1, 2, 3, 4, 5),
 'shots': 1000,
 'counts': {'000000': 368,
  '111111': 294,
  '011110': 6,
  '001000': 25,
  '111101': 17,
  '000100': 24,
  '111011': 26,
  '110111': 32,
  '000110': 2,
  '100001': 10,
  '111110': 28,
  '101111': 32,
  '000001': 24,
  '011000': 4,
  '010010': 2,
  '011111': 7,
  '011100': 6,
  '000010': 23,
  '010000': 18,
  '100011': 11,
  '101110': 1,
  '001001': 3,
  '111001': 3,
  '100111': 7,
  '110011': 4,
  '101011': 2,
  '011101': 1,
  '000101': 2,
  '110101': 2,
  '011001': 2,
  '010001': 1,
  '111010': 2,
  '001011': 1,
  '100000': 2,
  '110110': 1,
  '101101': 1,
  '010011': 1,
  '011011': 1,
  '010100': 1,
  '001100': 2,
  '001010': 1}}

In [9]:
obs = Observable(PauliString("X" * num_qubits))
print(obs)

X(q(0))*X(q(1))*X(q(2))*X(q(3))*X(q(4))*X(q(5))


In [10]:
from functools import partial

ideal_exec = partial(execute, rz_noise = 0.0, depolar_noise = 0.0)

ideal = obs.expectation(circuit, ideal_exec)
print("Ideal value:", "{:.5f}".format(ideal.real))

Ideal value: 1.00000


In [11]:
noisy_exec = partial(execute, rz_noise = 0.02, depolar_noise = 0.005)
noisy = obs.expectation(circuit, noisy_exec) 
print("Unmitigated noisy value:", "{:.5f}".format(noisy.real))

Unmitigated noisy value: 0.53600


In [12]:
from mitiq.ddd import insert_ddd_sequences, rules

print("Original circuit \n", circuit)

rule = rules.yy

ddd_circuit = insert_ddd_sequences(circuit, rule)
print("DDD modified circuit \n", ddd_circuit)

Original circuit 
                   ┌──┐           ┌──┐           ┌──┐           ┌──┐
0: ───H───@────────@──────────────@──────────────@──────────────@─────────
          │        │              │              │              │
1: ───────X───I────┼I────I───I────┼I────I───I────┼I────I───I────┼I────I───
                   │              │              │              │
2: ───I───I───I────X─────I───I────┼I────I───I────┼I────I───I────┼I────────
                                  │              │              │
3: ───I───I───I────I─────I───I────X─────I───I────┼I────I───I────┼I────────
                                                 │              │
4: ───I───I───I────I─────I───I────I─────I───I────X─────I───I────┼I────────
                                                                │
5: ───I───I───I────I─────I───I────I─────I───I────I─────I───I────X─────────
                  └──┘           └──┘           └──┘           └──┘
DDD modified circuit 
                   ┌──┐           ┌──┐     

In [13]:
ddd_noisy = obs.expectation(ddd_circuit, noisy_exec)

print("Unmitigated expectation value:", "{:.5f}".format(noisy.real))

print("Expectation value with DDD:", "{:.5f}".format(ddd_noisy.real))

Unmitigated expectation value: 0.53600
Expectation value with DDD: 0.79800


### Limitations with previous code 
- Digital DD with no Layout and ad-hoc noise parameters without any refrence to actual hardware noise parameters. 

**Proof of Concept of DD on cirq using noisy simulators**

### Using Qiskit code by mitiq


In [81]:
from collections.abc import Callable
import numpy as np
from matplotlib import pyplot as plt

import qiskit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch, SamplerOptions
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from mitiq.interface.mitiq_qiskit import to_qiskit
from mitiq import ddd, QPROGRAM
from mitiq.ddd import insert_ddd_sequences

In [82]:
import cirq

def rep_ixix_rule(window_length: int) -> Callable[[int], QPROGRAM]:
    return ddd.rules.repeated_rule(
        window_length, [cirq.I, cirq.X, cirq.I, cirq.X]
    )

def rep_xx_rule(window_length: int) -> Callable[[int], QPROGRAM]:
    return ddd.rules.repeated_rule(window_length, [cirq.X, cirq.X])

# Set DDD sequences to test.
rules = [rep_ixix_rule, rep_xx_rule, ddd.rules.xx]

# Test the sequence insertion
for rule in rules:
    print(rule(10))

0: ───I───I───X───I───X───I───X───I───X───I───
0: ───X───X───X───X───X───X───X───X───X───X───
0: ───I───I───I───X───I───I───X───I───I───I───


In [83]:
# Total number of shots to use.
shots = 10000

# Qubits to use on the experiment.
num_qubits = 2

# Test at multiple depths.
depths = [10, 30, 50, 100]

In [92]:
def get_circuit(depth: int):
    """Returns a circuit composed of a GHZ sequence, idle windows,
    and finally an inverse GHZ sequence.

    Args:
        depth: The depth of the idle window in the circuit.
    """
    circuit = qiskit.QuantumCircuit(num_qubits, num_qubits)
    circuit.h(0)
    circuit.cx(0, 1)
    for _ in range(depth):
        circuit.id(0)
        circuit.id(1)
    circuit.cx(0, 1)
    circuit.h(0)
    circuit.x(range(num_qubits))
    circuit.measure(0, 0)
    return circuit

In [93]:
ibm_circ = get_circuit(4)
print(ibm_circ)

     ┌───┐     ┌───┐┌───┐┌───┐┌───┐     ┌───┐┌───┐┌─┐
q_0: ┤ H ├──■──┤ I ├┤ I ├┤ I ├┤ I ├──■──┤ H ├┤ X ├┤M├
     └───┘┌─┴─┐├───┤├───┤├───┤├───┤┌─┴─┐├───┤└───┘└╥┘
q_1: ─────┤ X ├┤ I ├┤ I ├┤ I ├┤ I ├┤ X ├┤ X ├──────╫─
          └───┘└───┘└───┘└───┘└───┘└───┘└───┘      ║ 
c: 2/══════════════════════════════════════════════╩═
                                                   0 


In [94]:
ixix_circ = insert_ddd_sequences(ibm_circ, rep_ixix_rule)
print(ixix_circ)

     ┌───┐     ┌───┐┌───┐┌───┐┌───┐     ┌───┐┌───┐┌─┐
q_0: ┤ H ├──■──┤ I ├┤ X ├┤ I ├┤ X ├──■──┤ H ├┤ X ├┤M├
     └───┘┌─┴─┐├───┤├───┤├───┤├───┤┌─┴─┐├───┤└───┘└╥┘
q_1: ─────┤ X ├┤ I ├┤ X ├┤ I ├┤ X ├┤ X ├┤ X ├──────╫─
          └───┘└───┘└───┘└───┘└───┘└───┘└───┘      ║ 
c: 2/══════════════════════════════════════════════╩═
                                                   0 


In [95]:
USE_REAL_HARDWARE = False
correct_bitstring=[0]

In [96]:
if USE_REAL_HARDWARE:
    service = QiskitRuntimeService(name="qamp-2025")
    backend = service.least_busy(operational=True, simulator=False)
else:
    from qiskit_ibm_runtime.fake_provider import FakeMarrakesh 
    backend = FakeMarrakesh()


def ibm_executor(
    circuit: qiskit.QuantumCircuit,
    shots: int,
    correct_bitstring: list[int],
    noisy: bool = True,
) -> float:
    """Executes the input circuit(s) and returns ⟨A⟩, where 
    A = |correct_bitstring⟩⟨correct_bitstring| for each circuit.

    Args:
        circuit: Circuit to run.
        shots: Number of times to execute the circuit to compute the
            expectation value.
        correct_bitstring: Bitstring the circuit is expected to return, in the
            absence of noise.
    """
    if noisy:
        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=0,
        )
        transpiled = pm.run(circuit)

        if not isinstance(transpiled, list):
            transpiled = [transpiled]

        noise_model = NoiseModel.from_backend(backend)
        noisy_backend = AerSimulator(method='density_matrix',
                                noise_model=noise_model)
        job = noisy_backend.run(transpiled, shots=shots)
        
        all_counts = job.result().get_counts()
    else:
        ideal_backend = AerSimulator()
        job = ideal_backend.run(circuit, optimization_level=0, shots=shots)
        all_counts = job.result().get_counts()

    # Convert from raw measurement counts to the expectation value
    
    return all_counts

In [97]:
data = []
for depth in depths:
    circuit = get_circuit(depth)
    noisy_value = ibm_executor(
            circuit, shots=shots, correct_bitstring=correct_bitstring
    )
    data.append((depth, "unmitigated", noisy_value))
    for rule in rules:
        ddd_circuit = insert_ddd_sequences(circuit, rule)
        ddd_value = ibm_executor(
            ddd_circuit, shots=shots, correct_bitstring=correct_bitstring
        )
        data.append((depth, rule.__name__, ddd_value))

In [99]:
data

[(10, 'unmitigated', {'00': 155, '01': 9845}),
 (10, 'rep_ixix_rule', {'00': 176, '01': 9824}),
 (10, 'rep_xx_rule', {'00': 161, '01': 9839}),
 (10, 'xx', {'00': 185, '01': 9815}),
 (30, 'unmitigated', {'00': 265, '01': 9735}),
 (30, 'rep_ixix_rule', {'00': 268, '01': 9732}),
 (30, 'rep_xx_rule', {'00': 278, '01': 9722}),
 (30, 'xx', {'00': 272, '01': 9728}),
 (50, 'unmitigated', {'00': 316, '01': 9684}),
 (50, 'rep_ixix_rule', {'00': 339, '01': 9661}),
 (50, 'rep_xx_rule', {'00': 326, '01': 9674}),
 (50, 'xx', {'00': 347, '01': 9653}),
 (100, 'unmitigated', {'00': 507, '01': 9493}),
 (100, 'rep_ixix_rule', {'00': 539, '01': 9461}),
 (100, 'rep_xx_rule', {'00': 483, '01': 9517}),
 (100, 'xx', {'00': 543, '01': 9457})]